# Correcciones entrega 2

Este notebook aplica solamente la correccion de prefijos sobre `../data/processed/Argenprop_limpio.csv`:

- renombra las variables segun su origen: `original_`, `imputada_`, `sintetica_` o `enriquecida_`;
- no recalcula clusters ni ejecuta ningun algoritmo de clusterizacion;
- guarda el CSV limpio actualizado y un diccionario de variables.


In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

CLEAN_PATH = Path("../data/processed/Argenprop_limpio.csv")
DICT_PATH = Path("../data/processed/diccionario_variables_limpio.csv")

df = pd.read_csv(CLEAN_PATH, encoding="utf-8-sig")
print(df.shape)
df.head()


(7245, 55)


,Unnamed: 0,Precio,Expensas,Calle,Altura,Piso,Link,Ambientes,Dormitorios,Baños,...,Colegios_500m,Dist_Comisaria_m,Dist_Gimnasio_m,Dist_Supermercado_m,Supermercados_500m,Dist_Avenida_m,Avenida_cercana,Paradas_colectivo_300m,Antiguedad_imputada,Cluster
0,1,150000.0,260000.0,Bulnes,1600.0,NaN,https://www.argenprop.com/departamento-en-vent...,3,2,1.0,...,10,493.126679,51.727490,20.279331,10,156.788297,Avenida Santa Fe,19,0,2
1,2,330000.0,203300.0,ARAOZ,1200.0,8.0,https://www.argenprop.com/departamento-en-vent...,4,3,2.0,...,11,403.114757,429.584392,249.248104,4,96.514964,Avenida Raúl Scalabrini Ortiz,10,1,2
2,3,270000.0,300000.0,Honduras,3900.0,2.0,https://www.argenprop.com/departamento-en-vent...,4,2,2.0,...,7,891.145341,383.017854,75.243729,7,425.495375,Avenida Coronel Niceto Vega,5,0,2
3,4,570000.0,1000000.0,Castex,3300.0,NaN,https://www.argenprop.com/departamento-en-vent...,4,3,3.0,...,4,804.138281,286.697993,217.075594,3,108.444119,Avenida Casares,7,0,2
4,5,98000.0,150000.0,GURRUCHAGA,2100.0,5.0,https://www.argenprop.com/departamento-en-vent...,1,1,1.0,...,5,461.483469,393.568008,85.286744,6,378.802881,Avenida Raúl Scalabrini Ortiz,7,0,2


## Renombre por origen de variable

Criterio usado:

- `original_`: variables observadas directamente y no imputadas en el pipeline final.
- `imputada_`: variables que fueron completadas o imputadas durante limpieza.
- `sintetica_`: ids de exportacion y variables generadas previamente, como el cluster ya calculado.
- Los flags binarios de amenities y servicios se consideran `original_` porque provienen del aviso publicado.
- `enriquecida_`: variables agregadas por geocodificacion o cruces geoespaciales.


In [2]:
PREFIXES = ("original_", "imputada_", "sintetica_", "enriquecida_")

COLUMNAS_IMPUTADAS = {
    "Precio", "Expensas", "Piso", "Ambientes", "Dormitorios", "Antiguedad",
    "Estado", "Disposicion", "Tipo_Unidad", "Sup_Cubierta_m2", "Sup_Total_m2",
    "Antiguedad_imputada"
}

COLUMNAS_SINTETICAS = {"Unnamed_0", "id_registro", "Cluster", "cluster"}

COLUMNAS_ENRIQUECIDAS = {
    "Latitud", "Longitud", "Barrio", "Comuna", "Dist_Subte_m", "Subte_cercano",
    "Linea_subte", "Dist_Hospital_m", "Hospital_cercano", "Dist_Colegio_m",
    "Colegios_500m", "Dist_Comisaria_m", "Dist_Gimnasio_m", "Dist_Supermercado_m",
    "Supermercados_500m", "Dist_Avenida_m", "Avenida_cercana", "Paradas_colectivo_300m"
}

RENOMBRES_BASE = {"Unnamed: 0": "id_registro", "Unnamed_0": "id_registro"}

def normalizar_nombre(columna):
    nombre = RENOMBRES_BASE.get(str(columna), str(columna))
    nombre = unicodedata.normalize("NFKD", nombre)
    nombre = nombre.encode("ascii", "ignore").decode("ascii")
    nombre = re.sub(r"[^0-9A-Za-z_]+", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre).strip("_")
    return nombre

def tipo_variable(columna):
    nombre = normalizar_nombre(columna)
    if nombre in COLUMNAS_IMPUTADAS or nombre.endswith("_imputada"):
        return "imputada"
    if nombre in COLUMNAS_ENRIQUECIDAS:
        return "enriquecida"
    if nombre in COLUMNAS_SINTETICAS:
        return "sintetica"
    return "original"

def nombre_original_desde_columna(columna):
    columna = str(columna)
    if columna.startswith(PREFIXES):
        return columna.split("_", 1)[1]
    return columna

def nombre_prefijado(columna):
    nombre_original = nombre_original_desde_columna(columna)
    return f"{tipo_variable(nombre_original)}_{normalizar_nombre(nombre_original)}"

# Si una variable ya existe prefijada y tambien aparece sin prefijo, se conserva la version sin prefijo
# para poder incorporar columnas agregadas despues, como Cluster, sin recalcular nada.
columnas_a_eliminar = []
for col in df.columns:
    if not str(col).startswith(PREFIXES):
        destino = nombre_prefijado(col)
        if destino in df.columns:
            columnas_a_eliminar.append(destino)

if columnas_a_eliminar:
    print("Columnas prefijadas reemplazadas por version sin prefijo:", columnas_a_eliminar)
    df = df.drop(columns=columnas_a_eliminar)

nombres_originales = [nombre_original_desde_columna(col) for col in df.columns]
renombres = {col: nombre_prefijado(col) for col in df.columns}

if len(set(renombres.values())) != len(renombres):
    duplicados = pd.Series(list(renombres.values())).value_counts()
    duplicados = duplicados[duplicados > 1].index.tolist()
    raise ValueError(f"Hay nombres duplicados luego de renombrar: {duplicados}")

diccionario = pd.DataFrame({
    "variable_original": nombres_originales,
    "tipo_variable": [tipo_variable(nombre_original) for nombre_original in nombres_originales],
    "variable_renombrada": [renombres[col] for col in df.columns]
})

df_renombrado = df.rename(columns=renombres)

diccionario["tipo_variable"].value_counts().sort_index()


tipo_variable
enriquecida    18
imputada       12
original       23
sintetica       2
Name: count, dtype: int64

In [3]:
df_renombrado.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
diccionario.to_csv(DICT_PATH, index=False, encoding="utf-8-sig")

print(f"Archivo actualizado: {CLEAN_PATH}")
print(f"Diccionario generado: {DICT_PATH}")
print("Shape:", df_renombrado.shape)
df_renombrado.head()


Archivo actualizado: Argenprop_limpio.csv
Diccionario generado: diccionario_variables_limpio.csv
Shape: (7245, 55)


,sintetica_id_registro,imputada_Precio,imputada_Expensas,original_Calle,original_Altura,imputada_Piso,original_Link,imputada_Ambientes,imputada_Dormitorios,original_Banos,...,enriquecida_Colegios_500m,enriquecida_Dist_Comisaria_m,enriquecida_Dist_Gimnasio_m,enriquecida_Dist_Supermercado_m,enriquecida_Supermercados_500m,enriquecida_Dist_Avenida_m,enriquecida_Avenida_cercana,enriquecida_Paradas_colectivo_300m,imputada_Antiguedad_imputada,sintetica_Cluster
0,1,150000.0,260000.0,Bulnes,1600.0,NaN,https://www.argenprop.com/departamento-en-vent...,3,2,1.0,...,10,493.126679,51.727490,20.279331,10,156.788297,Avenida Santa Fe,19,0,2
1,2,330000.0,203300.0,ARAOZ,1200.0,8.0,https://www.argenprop.com/departamento-en-vent...,4,3,2.0,...,11,403.114757,429.584392,249.248104,4,96.514964,Avenida Raúl Scalabrini Ortiz,10,1,2
2,3,270000.0,300000.0,Honduras,3900.0,2.0,https://www.argenprop.com/departamento-en-vent...,4,2,2.0,...,7,891.145341,383.017854,75.243729,7,425.495375,Avenida Coronel Niceto Vega,5,0,2
3,4,570000.0,1000000.0,Castex,3300.0,NaN,https://www.argenprop.com/departamento-en-vent...,4,3,3.0,...,4,804.138281,286.697993,217.075594,3,108.444119,Avenida Casares,7,0,2
4,5,98000.0,150000.0,GURRUCHAGA,2100.0,5.0,https://www.argenprop.com/departamento-en-vent...,1,1,1.0,...,5,461.483469,393.568008,85.286744,6,378.802881,Avenida Raúl Scalabrini Ortiz,7,0,2
